# Laboratorio Clase 8: ¿automatización o agente?

**Módulo II: Navegando por los componentes de la IA. Sesión 8, 17 de noviembre de 2026**

En la clase vimos una regla simple: si puedes escribir los `if`, escríbelos. Un agente vale la pena cuando hay ambigüedad, razonamiento contextual o excepciones frecuentes. En este notebook resuelves los mismos tickets de atención al cliente de dos maneras y comparas el resultado.

| Misión | Qué vas a hacer | Concepto de la clase |
|---|---|---|
| 1 | Aplicar la regla de elegibilidad a tres procesos | ¿Agente o automatización? |
| 2 | Construir una automatización con un componente agéntico | El modelo clasifica, las reglas deciden |
| 3 | Construir un agente con autonomía controlada | Loop del harness, herramientas y guardrails |
| 4 | Comparar ambos caminos con los mismos tickets | Costo, control y flexibilidad |
| 5 | Probar el kill switch y la aprobación humana | HITL y botón de parada |

HITL significa *human in the loop*: un humano aprueba las acciones riesgosas. Un *guardrail* es un límite que el código impone al agente, no una sugerencia en el prompt.

### Cómo usar este notebook

Ejecuta cada celda con **Shift + Enter**. Lee el resultado antes de seguir. Si algo falla, usa **Kernel, Restart y Run All**. No necesitas entender cada línea: lee los comentarios que empiezan con `#`.

Cada misión termina con una pregunta **Para la puesta en común**. Anota tu respuesta: la conversamos al final.

## Paso 0: revisar el ambiente

Este laboratorio tiene su propio ambiente virtual en la carpeta `clase_08-agentes/.venv`. Se crea con `uv sync`. Las misiones 2 y 3 usan la API de Claude si encuentran la variable `ANTHROPIC_API_KEY` en un archivo `.env`. Copia `env.example` como `.env` y pega tu clave. Sin clave, el notebook usa un modo simulado que sigue un guion fijo.

In [ ]:
# Paso 0: importar las herramientas y buscar la clave
import importlib.util, subprocess, sys
if "google.colab" in sys.modules and importlib.util.find_spec("anthropic") is None:
    subprocess.run(["uv", "pip", "install", "--system", "-q", "anthropic"], check=True)  # Colab no trae el SDK

import json, os, re
from pathlib import Path
from types import SimpleNamespace
from typing import Literal

import anthropic
from pydantic import BaseModel

USAR_CLAUDE = True  # cambia a False para usar el modo simulado
MODELO = "claude-haiku-5-5"

def cargar_env():
    try:
        from dotenv import load_dotenv
    except ImportError:   # por ejemplo, en Colab
        return False
    aqui = Path.cwd().resolve()
    for carpeta in [aqui, *list(aqui.parents)[:5]]:
        env = carpeta / ".env"
        if env.is_file():
            load_dotenv(env); return True
    return False

if USAR_CLAUDE:
    cargar_env()
hay_clave = USAR_CLAUDE and bool(os.environ.get("ANTHROPIC_API_KEY"))
print("Modo:", "Claude Haiku 5.5 por API" if hay_clave else "simulado (sin clave de API)")

### Los datos: Tienda Andina

Tienda Andina vende por internet. Recibe tickets de clientes que piden devoluciones, reclaman despachos o reportan cobros duplicados. Estos datos son ficticios.

In [ ]:
# Paso 0b: pedidos, tickets y políticas de atención
PEDIDOS = {
    "A-1001": {"producto": "Hervidor eléctrico", "monto": 25_000, "dias_desde_compra": 3, "estado": "entregado", "vip": False},
    "A-1002": {"producto": "Audífonos inalámbricos", "monto": 40_000, "dias_desde_compra": 12, "estado": "en tránsito", "vip": False},
    "A-1003": {"producto": "Polerón", "monto": 18_000, "dias_desde_compra": 5, "estado": "entregado", "vip": False},
    "A-1004": {"producto": "Chaqueta outdoor", "monto": 120_000, "dias_desde_compra": 20, "estado": "entregado", "vip": True},
    "A-1005": {"producto": "Notebook 14 pulgadas", "monto": 450_000, "dias_desde_compra": 25, "estado": "entregado", "vip": False},
}

TICKETS = [
    {"id": "T1", "pedido_id": "A-1001", "texto": "El hervidor llegó con la tapa rota. Quiero la devolución de mi dinero."},
    {"id": "T2", "pedido_id": "A-1002", "texto": "Compré audífonos hace 12 días y todavía no llegan. ¿Me pueden devolver la plata?"},
    {"id": "T3", "pedido_id": "A-1003", "texto": "Me cobraron dos veces el polerón en la tarjeta."},
    {"id": "T4", "pedido_id": "A-1004", "texto": "La chaqueta llegó tarde y me perdí el viaje para el que la compré. Además el cierre se traba. "
                                               "No sé si prefiero cambiarla por otra talla o que me devuelvan una parte. Soy cliente hace años."},
    {"id": "T5", "pedido_id": "A-1005", "texto": "El notebook se apaga solo. Un ejecutivo me prometió por teléfono el reembolso total. Exijo que lo hagan hoy."},
]

POLITICAS = {
    "devolucion": "Producto dañado o defectuoso: reembolso total dentro de 30 días desde la compra.",
    "despacho": "Pedido no entregado después de 10 días: reembolso total o reenvío, a elección del cliente.",
    "cobro_duplicado": "Cobro duplicado: se reversa el segundo cargo sin revisión adicional.",
    "compensacion": "Cliente VIP con mala experiencia: se ofrece cambio sin costo o un reembolso parcial de hasta 30%.",
    "promesas": "Las promesas verbales no obligan a la empresa. Se registran y se escalan a un supervisor.",
}
print(f"{len(TICKETS)} tickets y {len(PEDIDOS)} pedidos cargados.")

## Misión 1: ¿agente o automatización?

La regla de la clase tiene tres preguntas. ¿Hay ambigüedad en la entrada? ¿Se necesita razonamiento contextual? ¿Las excepciones son frecuentes? Con dos o más respuestas SÍ, conviene un agente. Con todas NO, conviene una automatización. Con una sola, conviene un híbrido.

In [ ]:
# Misión 1a: la regla de elegibilidad como función
def elegir_enfoque(ambiguedad: bool, razonamiento_contextual: bool, excepciones_frecuentes: bool) -> str:
    si = ambiguedad + razonamiento_contextual + excepciones_frecuentes   # True cuenta como 1
    if si >= 2:
        return "Agente"
    if si == 0:
        return "Automatización"
    return "Híbrido: el agente decide, la automatización ejecuta, un humano verifica"

PROCESOS = {
    "Triage de tickets de atención": (True, True, True),
    "Reembolsos estándar bajo $50.000": (False, False, False),
    "Conciliación bancaria con diferencias": (False, True, False),
}
for proceso, respuestas in PROCESOS.items():
    print(f"{proceso:40} -> {elegir_enfoque(*respuestas)}")

In [ ]:
# Misión 1b: agrega un proceso de tu empresa y responde las tres preguntas
mi_proceso = "Aprobar notas de crédito"           # cambia el nombre
mis_respuestas = (False, True, True)            # (ambigüedad, razonamiento contextual, excepciones frecuentes)
print(f"{mi_proceso} -> {elegir_enfoque(*mis_respuestas)}")

**Para la puesta en común:** ¿qué proceso de tu empresa parece candidato a agente, pero en realidad cabe en cinco reglas `if`? ¿Cuánto cuesta cada camino si el proceso corre 10.000 veces al mes?

---

## Misión 2: automatización con un componente agéntico

El camino (a). El modelo hace una sola tarea acotada: leer el texto libre del ticket y entregar una clasificación con un formato fijo. Después, reglas escritas en Python deciden qué hacer. El modelo no ejecuta nada.

In [ ]:
# Misión 2a: el formato que exigimos al modelo
class Clasificacion(BaseModel):
    categoria: Literal["producto_dañado", "no_entregado", "cobro_duplicado", "otro"]
    pide: Literal["reembolso", "cambio", "no_claro"]
    tono: Literal["neutro", "molesto", "muy_molesto"]

def clasificar_simulado(ticket):
    # ponytail: palabras clave fijas para clases sin clave; un modelo real entiende sinónimos
    t = ticket["texto"].lower()
    if "dos veces" in t:
        categoria = "cobro_duplicado"
    elif any(p in t for p in ("rota", "roto", "falla", "apaga")):
        categoria = "producto_dañado"
    elif "no llega" in t:
        categoria = "no_entregado"
    else:
        categoria = "otro"
    devolver, cambiar = any(p in t for p in ("devol", "devuelvan", "reembolso")), "cambi" in t
    pide = "no_claro" if devolver == cambiar else ("reembolso" if devolver else "cambio")
    tono = "muy_molesto" if "exijo" in t else ("molesto" if "?" in t or "rota" in t else "neutro")
    return Clasificacion(categoria=categoria, pide=pide, tono=tono)

def clasificar(ticket):
    if not hay_clave:
        return clasificar_simulado(ticket)
    try:
        respuesta = anthropic.Anthropic().messages.parse(
            model=MODELO,
            max_tokens=512,
            system="Clasificas tickets de atención de una tienda chilena. Si el cliente pide dos cosas distintas, usa pide='no_claro'.",
            messages=[{"role": "user", "content": ticket["texto"]}],
            output_format=Clasificacion,
        )
    except anthropic.APIError as error:   # sin internet, clave inválida o servicio caído
        print(f"La API falló ({type(error).__name__}). Sigo en modo simulado.")
        return clasificar_simulado(ticket)
    return respuesta.parsed_output or clasificar_simulado(ticket)

for ticket in TICKETS:
    print(ticket["id"], clasificar(ticket).model_dump())

In [ ]:
# Misión 2b: reglas fijas deciden la acción; el modelo ya no participa
MONTO_MAX_AUTOMATICO = 50_000

def decidir(c: Clasificacion, pedido: dict) -> tuple[str, str]:
    if c.categoria == "cobro_duplicado":
        return "reversar_cobro", "Política de cobro duplicado"
    if c.pide != "reembolso":
        return "derivar_a_humano", "El cliente no pide un reembolso claro"
    if pedido["monto"] > MONTO_MAX_AUTOMATICO:
        return "derivar_a_humano", f"Monto sobre ${MONTO_MAX_AUTOMATICO:,}".replace(",", ".")
    if c.categoria == "producto_dañado" and pedido["dias_desde_compra"] <= 30:
        return "reembolso_total", "Producto dañado dentro de 30 días"
    if c.categoria == "no_entregado" and pedido["dias_desde_compra"] > 10 and pedido["estado"] != "entregado":
        return "reembolso_total", "Despacho atrasado más de 10 días"
    return "derivar_a_humano", "Ninguna regla aplica"

def atender_con_automatizacion(ticket):
    return decidir(clasificar(ticket), PEDIDOS[ticket["pedido_id"]])

for ticket in TICKETS:
    accion, motivo = atender_con_automatizacion(ticket)
    print(f"{ticket['id']}: {accion:18} ({motivo})")

La automatización resuelve sola los casos estándar y es predecible: el mismo ticket siempre recibe la misma acción. Los tickets T4 y T5 terminan en un humano porque ninguna regla los cubre.

**Para la puesta en común:** ¿qué regla agregarías para resolver T4 sin un humano? ¿Cuántas reglas más necesitarías si llegan 50 variantes de T4 al mes?

---

## Misión 3: un agente con autonomía controlada

El camino (b). Ahora el modelo elige qué herramientas usar y en qué orden. El código mantiene el control con cinco guardrails:

1. **Lista de herramientas permitidas:** el agente solo puede llamar lo que está en la lista.
2. **Límite de pasos:** después de `MAX_PASOS` vueltas, el caso se escala.
3. **Aprobación humana (HITL):** un reembolso sobre `UMBRAL_HITL` requiere la firma de un supervisor.
4. **Kill switch:** una variable detiene al agente antes del siguiente paso.
5. **Bitácora:** cada llamada queda registrada para la auditoría.

In [ ]:
# Misión 3a: los contratos de las herramientas (nombre, descripción y esquema de entrada)
def herramienta(nombre, descripcion, **propiedades):
    return {"name": nombre, "description": descripcion, "strict": True,
            "input_schema": {"type": "object", "properties": propiedades,
                             "required": list(propiedades), "additionalProperties": False}}

TEXTO = {"type": "string"}
ENTERO = {"type": "integer"}
HERRAMIENTAS = [
    herramienta("consultar_pedido", "Devuelve producto, monto, días desde la compra, estado y si el cliente es VIP.", pedido_id=TEXTO),
    herramienta("consultar_politica", "Devuelve el texto de una política de atención.",
                tema={"type": "string", "enum": list(POLITICAS)}),
    herramienta("emitir_reembolso", "Emite un reembolso en pesos. Acción irreversible. Sobre el umbral requiere aprobación humana.",
                pedido_id=TEXTO, monto=ENTERO, motivo=TEXTO),
    herramienta("ofrecer_cambio", "Ofrece al cliente un cambio de producto sin costo.", pedido_id=TEXTO, detalle=TEXTO),
    herramienta("escalar_a_humano", "Deriva el caso a un supervisor con un resumen.", motivo=TEXTO),
]

SISTEMA_AGENTE = (
    "Eres el agente de atención de Tienda Andina. Resuelve el ticket con las herramientas. "
    "Revisa siempre el pedido y la política antes de actuar. No inventes datos. "
    "Si una herramienta rechaza la acción, escala el caso. Termina con un mensaje breve para el cliente."
)
print("Herramientas:", [h["name"] for h in HERRAMIENTAS])

In [ ]:
# Misión 3b: los guardrails viven en el código, no en el prompt
MAX_PASOS = 6
UMBRAL_HITL = 100_000
KILL_SWITCH = False         # True detiene al agente antes del siguiente paso
SUPERVISOR_APRUEBA = False  # simula la respuesta del supervisor; en producción es una cola de aprobación
PERMITIDAS = {h["name"] for h in HERRAMIENTAS}
bitacora = []

def pesos(monto):
    return f"${monto:,}".replace(",", ".")

def emitir_reembolso(pedido_id, monto, motivo):
    pedido = PEDIDOS[pedido_id]
    if monto > pedido["monto"]:
        raise ValueError(f"El monto {pesos(monto)} supera el valor del pedido.")
    if monto > UMBRAL_HITL and not SUPERVISOR_APRUEBA:
        return f"RECHAZADO: un reembolso sobre {pesos(UMBRAL_HITL)} requiere aprobación humana y el supervisor no aprobó."
    return f"Reembolso de {pesos(monto)} emitido para {pedido_id}. Motivo: {motivo}."

ACCIONES = {
    "consultar_pedido": lambda pedido_id: json.dumps(PEDIDOS[pedido_id], ensure_ascii=False),
    "consultar_politica": lambda tema: POLITICAS[tema],
    "emitir_reembolso": emitir_reembolso,
    "ofrecer_cambio": lambda pedido_id, detalle: f"Cambio ofrecido para {pedido_id}: {detalle}.",
    "escalar_a_humano": lambda motivo: f"Caso escalado a supervisor. Motivo: {motivo}.",
}

def ejecutar_herramienta(ticket_id, nombre, entrada):
    """Devuelve (texto, es_error) y deja la llamada en la bitácora."""
    if nombre not in PERMITIDAS:
        texto, error = f"Herramienta no permitida: {nombre}", True
    else:
        try:
            texto, error = ACCIONES[nombre](**entrada), False
        except (KeyError, TypeError, ValueError) as e:
            texto, error = f"Error: {e}", True
    bitacora.append({"ticket": ticket_id, "herramienta": nombre, "entrada": entrada, "resultado": texto, "error": error})
    return texto, error

In [ ]:
# Misión 3c: el loop del harness (pensar, llamar herramienta, observar, repetir)
def atender_con_agente(ticket, cliente):
    mensajes = [{"role": "user", "content": f"Ticket {ticket['id']}, pedido {ticket['pedido_id']}:\n{ticket['texto']}"}]
    for paso in range(1, MAX_PASOS + 1):
        if KILL_SWITCH:
            bitacora.append({"ticket": ticket["id"], "herramienta": "kill_switch", "entrada": {}, "resultado": "detenido", "error": False})
            return "Agente detenido por kill switch. Un humano toma el caso.", paso - 1
        respuesta = cliente.messages.create(model=MODELO, max_tokens=1024, system=SISTEMA_AGENTE,
                                            tools=HERRAMIENTAS, messages=mensajes)
        mensajes.append({"role": "assistant", "content": respuesta.content})
        if respuesta.stop_reason != "tool_use":
            return next((b.text for b in respuesta.content if b.type == "text"), ""), paso
        resultados = []
        for bloque in respuesta.content:
            if bloque.type == "tool_use":
                texto, error = ejecutar_herramienta(ticket["id"], bloque.name, bloque.input)
                resultados.append({"type": "tool_result", "tool_use_id": bloque.id, "content": texto, "is_error": error})
        mensajes.append({"role": "user", "content": resultados})   # todos los resultados en un solo mensaje
    ejecutar_herramienta(ticket["id"], "escalar_a_humano", {"motivo": f"Se alcanzó el límite de {MAX_PASOS} pasos"})
    return f"Límite de {MAX_PASOS} pasos alcanzado. Caso escalado a un humano.", MAX_PASOS

In [ ]:
# Misión 3d: un cliente simulado que sigue un guion fijo cuando no hay clave
GUION = {   # ponytail: decisiones escritas a mano; con clave, Claude las toma solo
    "T1": [("consultar_politica", {"tema": "devolucion"}), ("emitir_reembolso", {"pedido_id": "A-1001", "monto": 25_000, "motivo": "Producto dañado"})],
    "T2": [("consultar_politica", {"tema": "despacho"}), ("emitir_reembolso", {"pedido_id": "A-1002", "monto": 40_000, "motivo": "Despacho atrasado"})],
    "T3": [("consultar_politica", {"tema": "cobro_duplicado"}), ("emitir_reembolso", {"pedido_id": "A-1003", "monto": 18_000, "motivo": "Cobro duplicado"})],
    "T4": [("consultar_politica", {"tema": "compensacion"}), ("ofrecer_cambio", {"pedido_id": "A-1004", "detalle": "Chaqueta de otra talla sin costo, o reembolso parcial de 30%"})],
    "T5": [("consultar_politica", {"tema": "promesas"}), ("emitir_reembolso", {"pedido_id": "A-1005", "monto": 450_000, "motivo": "Producto defectuoso"}),
           ("escalar_a_humano", {"motivo": "Reembolso sobre el umbral y promesa verbal de un ejecutivo"})],
}

class ClienteSimulado:
    def __init__(self):
        self.messages = self

    def create(self, messages, **_):
        ticket_id, pedido_id = re.match(r"Ticket (\w+), pedido ([\w-]+)", messages[0]["content"]).groups()
        pasos = [("consultar_pedido", {"pedido_id": pedido_id}), *GUION[ticket_id]]
        k = sum(m["role"] == "assistant" for m in messages)
        ultimo = messages[-1]["content"][-1]["content"] if k else ""
        if k < len(pasos) and not (pasos[k][0] == "escalar_a_humano" and not ultimo.startswith("RECHAZADO")):
            nombre, entrada = pasos[k]
            bloque = SimpleNamespace(type="tool_use", id=f"sim_{k}", name=nombre, input=entrada)
            return SimpleNamespace(stop_reason="tool_use", content=[bloque])
        return SimpleNamespace(stop_reason="end_turn", content=[SimpleNamespace(type="text", text=f"Estimado cliente: {ultimo}")])

cliente = anthropic.Anthropic() if hay_clave else ClienteSimulado()

In [ ]:
# Misión 3e: el agente atiende el ticket ambiguo T4
def atender(ticket):
    global cliente
    try:
        return atender_con_agente(ticket, cliente)
    except anthropic.APIError as error:
        print(f"La API falló ({type(error).__name__}). Sigo en modo simulado.")
        cliente = ClienteSimulado()
        return atender_con_agente(ticket, cliente)

bitacora.clear()
respuesta, pasos = atender(TICKETS[3])
print(f"Pasos: {pasos}\nRespuesta: {respuesta}\n")
for fila in bitacora:
    print(f"  {fila['herramienta']:20} {json.dumps(fila['entrada'], ensure_ascii=False)}")

**Para la puesta en común:** el agente atendió T4 sin una regla escrita para ese caso. Lee su bitácora: ¿qué consultó antes de actuar? ¿Qué guardrail evitó que regalara un reembolso total? ¿Qué pasaría sin la lista de herramientas permitidas?

---

## Misión 4: los mismos tickets por ambos caminos

In [ ]:
# Misión 4a: comparación lado a lado
bitacora.clear()
print(f"{'Ticket':7}{'Automatización':20}{'Agente':52}{'Llamadas LLM (a / b)'}")
for ticket in TICKETS:
    accion, _ = atender_con_automatizacion(ticket)
    respuesta, pasos = atender(ticket)
    print(f"{ticket['id']:7}{accion:20}{respuesta[:48]:52}1 / {pasos}")

In [ ]:
# Misión 4b: la bitácora completa del agente, lista para auditoría
for fila in bitacora:
    marca = "ERROR" if fila["error"] else "ok"
    print(f"{fila['ticket']} {fila['herramienta']:20} {marca:5} {fila['resultado'][:70]}")

La automatización usa una llamada al modelo por ticket. El agente usa varias, una por vuelta del loop, y cada llamada suma costo y latencia. A cambio, el agente atiende T4 con contexto y deja T5 en manos de un supervisor con contexto.

**Para la puesta en común:** diseña un híbrido para Tienda Andina. ¿Qué tickets van por el camino (a) y cuáles por el (b)? ¿Quién decide la ruta: una regla o un clasificador?

---

## Misión 5: kill switch y aprobación humana

In [ ]:
# Misión 5a: el kill switch detiene al agente antes de su primer paso
KILL_SWITCH = True
print(atender(TICKETS[0]))
KILL_SWITCH = False

In [ ]:
# Misión 5b: el supervisor aprueba el reembolso de T5
SUPERVISOR_APRUEBA = True
bitacora.clear()
respuesta, pasos = atender(TICKETS[4])
print(f"Pasos: {pasos}\nRespuesta: {respuesta}")
SUPERVISOR_APRUEBA = False

**Para la puesta en común:** ¿quién debe poder activar el kill switch en tu empresa y en qué condiciones? ¿Qué monto usarías como `UMBRAL_HITL` y por qué?

---

## Cierre

Hoy resolviste los mismos tickets de dos maneras:

1. **Regla de elegibilidad:** dos o más SÍ apuntan a un agente; ningún SÍ apunta a una automatización.
2. **Automatización con un componente agéntico:** el modelo clasifica y las reglas deciden. Es barata, predecible y auditable.
3. **Agente con autonomía controlada:** el modelo elige las herramientas y el código pone los límites.
4. **Guardrails:** lista permitida, límite de pasos, HITL, kill switch y bitácora.
5. **Decisión:** usa el agente donde las reglas no alcanzan, y la automatización en todo lo demás.

En la clase 9 vamos a diseñar agentes con patrones y a observarlos con trazas en LangSmith.